<a href="https://colab.research.google.com/github/HafizRizqi/MachineLearning_2026/blob/main/JS09/JS09_Tugas_Lab_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#🥎Tugas Lab 2

Lakukan percobaan penggunaan ANNOY, FAISS, dan HNSWLIB pada dataset sekunder berukuran besar (Micro Spotify) pada link berikut: https://www.kaggle.com/datasets/bwandowando/spotify-songs-with-attributes-and-lyrics/data .

Download data dan load CSV filenya (pilih dataset yg pertama dari dua dataset).

Pilih hanya fitur numerik saja, dan lakukan normalisasi menggunakan StandardScaler.

Lakukan pencarian track terdekat dan bandingkan hasilnya.

Lakkan perbandingan pada exact NN, ANNOY, FAISS, dan HNSW

Langkah 1 – Mount Google Drive

In [1]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


Langkah 2 – Ambil dataset

In [2]:
import os

file_path = '/content/drive/MyDrive/MachineLearning/songs_with_attributes_and_lyrics.csv'

if os.path.exists(file_path):
    print("Dataset berhasil ditemukan!")
    print("Ukuran file:", round(os.path.getsize(file_path) / (1024**3), 2), "GB")
else:
    print("Dataset tidak ditemukan. Periksa lokasi file.")

Dataset berhasil ditemukan!
Ukuran file: 1.44 GB


Langkah 3 - Instalasi library

In [3]:
!pip install -q annoy faiss-cpu hnswlib

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


Langkah 4 – Import library

In [4]:
import pandas as pd
import numpy as np
import time
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors

from annoy import AnnoyIndex
import faiss
import hnswlib

Langkah 5 – Membaca dataset dan Memeriksa nama kolom datase

In [5]:
header = pd.read_csv(file_path, nrows=0)

print("Daftar kolom dataset:")
print(header.columns.tolist())

print("\nJumlah kolom:", len(header.columns))

Daftar kolom dataset:
['id', 'name', 'album_name', 'artists', 'danceability', 'energy', 'key', 'loudness', 'mode', 'speechiness', 'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo', 'duration_ms', 'lyrics']

Jumlah kolom: 17


Langkah 6 – Memilih fitur numerik

In [6]:
features = [
    'danceability',
    'energy',
    'loudness',
    'speechiness',
    'acousticness',
    'instrumentalness',
    'liveness',
    'valence',
    'tempo'
]

# Memeriksa apakah semua fitur tersedia
available_columns = header.columns.tolist()

missing_features = [
    col for col in features
    if col not in available_columns
]

if missing_features:
    raise ValueError(
        f"Kolom tidak ditemukan: {missing_features}"
    )

# Kolom tambahan untuk identitas lagu
metadata = [
    col for col in ['track_name', 'track_id', 'artists']
    if col in available_columns
]

# Membaca hanya fitur numerik dan metadata
df = pd.read_csv(
    file_path,
    usecols=features + metadata,
    low_memory=False
)

print("Dataset berhasil dibaca!")
print("Ukuran dataset:", df.shape)
display(df.head())

Dataset berhasil dibaca!
Ukuran dataset: (955320, 10)


,artists,danceability,energy,loudness,speechiness,acousticness,instrumentalness,liveness,valence,tempo
0,HELLYEAH,0.415,0.6050,-11.157,0.0575,0.00116,0.838000,0.4710,0.193,100.059
1,Yxngxr1,0.788,0.6480,-9.135,0.3150,0.90000,0.000000,0.1760,0.287,79.998
2,Glowie,0.000,0.0354,-20.151,0.0000,0.90800,0.000000,0.4790,0.000,0.000
3,Rosendo,0.657,0.8820,-6.340,0.0385,0.00740,0.000013,0.0474,0.939,123.588
4,Rosendo,0.659,0.8930,-8.531,0.0411,0.09220,0.000019,0.0534,0.951,123.600


Langkah 7 – Menghapus data kosong dan tidak valid

In [7]:
# Mencatat jumlah data awal
initial_rows = len(df)

print("Jumlah data sebelum dibersihkan:", initial_rows)

# Mengubah fitur menjadi numerik
df[features] = df[features].apply(
    pd.to_numeric,
    errors='coerce'
)

# Menghitung data kosong
missing_before = df[features].isna().any(axis=1).sum()

# Menghapus baris yang memiliki nilai kosong
df = df.dropna(subset=features)

# Menghapus data infinity
X_temp = df[features].to_numpy(dtype=np.float32)

valid_rows = np.isfinite(X_temp).all(axis=1)

df = df.loc[valid_rows].reset_index(drop=True)

# Mencatat jumlah data akhir
final_rows = len(df)

# Menghitung jumlah data yang dihapus
removed_rows = initial_rows - final_rows

print("\nHASIL PEMBERSIHAN DATA")
print("Data awal       :", initial_rows)
print("Data kosong     :", missing_before)
print("Data dihapus    :", removed_rows)
print("Data akhir      :", final_rows)

print(
    "Persentase data tersisa:",
    round(final_rows / initial_rows * 100, 2),
    "%"
)

Jumlah data sebelum dibersihkan: 955320

HASIL PEMBERSIHAN DATA
Data awal       : 955320
Data kosong     : 0
Data dihapus    : 0
Data akhir      : 955320
Persentase data tersisa: 100.0 %


Tidak Terdapat Data yang kosong (NaN) ataupun Tidak Valid (-infinity)

Langkah 8 – Melakukan normalisasi menggunakan StandardScaler

In [8]:
# Mengambil fitur numerik
X = df[features].to_numpy(dtype=np.float32)

# Membuat objek StandardScaler
scaler = StandardScaler()

# Melakukan standardisasi
X_scaled = scaler.fit_transform(X)

# Memastikan tipe data float32 dan tersimpan berurutan di memori
X_scaled = np.ascontiguousarray(
    X_scaled,
    dtype=np.float32
)

print("Normalisasi berhasil!")
print("Ukuran data:", X_scaled.shape)
print("\nHasil normalisasi:")
print(X_scaled[:5])

Normalisasi berhasil!
Ukuran data: (955320, 9)

Hasil normalisasi:
[[-0.799307   -0.19864804 -0.87638533 -0.28127217 -0.9037919   3.5533972
   1.280052   -1.1735829  -0.7505038 ]
 [ 1.3975976  -0.01859904 -0.34315994  2.4896579   1.9789547  -0.38476917
  -0.2255298  -0.799778   -1.429702  ]
 [-3.2435842  -2.5836701  -3.2482107  -0.9000235   2.0046122  -0.38476917
   1.3208814  -1.9410762  -4.138167  ]
 [ 0.6260307   0.961203    0.39391503 -0.48572916 -0.88377905 -0.3847095
  -0.8818614   1.7929972   0.04610954]
 [ 0.6378102   1.0072619  -0.18387789 -0.45775083 -0.61180973 -0.38467944
  -0.8512394   1.840717    0.04651586]]


Langkah 9 – Menentukan jumlah tetangga terdekat

In [9]:
k = 10

# Jumlah query
n_query = 10

# Memilih 10 lagu secara acak sebagai query
np.random.seed(42)

query_indices = np.random.choice(
    len(X_scaled),
    size=n_query,
    replace=False
)

# Mengambil vektor query
Q = X_scaled[query_indices]

print("Jumlah query:", len(Q))
print("Jumlah nearest neighbors:", k)

Jumlah query: 10
Jumlah nearest neighbors: 10


Langkah 10 – Membuat pencarian Exact NN

In [10]:
# Membuat model Exact NN
exact_nn = NearestNeighbors(
    n_neighbors=k + 1,
    algorithm='brute',
    metric='euclidean',
    n_jobs=-1
)

# Menghitung waktu pembangunan model
start = time.perf_counter()

exact_nn.fit(X_scaled)

exact_build_time = time.perf_counter() - start

# Melakukan pencarian
start = time.perf_counter()

exact_distances, exact_indices = exact_nn.kneighbors(Q)

exact_query_time = (
    time.perf_counter() - start
) / n_query

print("Exact NN berhasil dijalankan!")
print("Build/Fit Time:", round(exact_build_time, 4), "detik")
print("Query Time:", round(exact_query_time * 1000, 4), "ms/query")

Exact NN berhasil dijalankan!
Build/Fit Time: 0.0343 detik
Query Time: 57.0846 ms/query


Langkah 11 – Membuat index ANNOY

In [11]:
dim = X_scaled.shape[1]

# Membuat index ANNOY dengan Euclidean Distance
annoy_index = AnnoyIndex(dim, 'euclidean')

# Menghitung waktu pembangunan index
start = time.perf_counter()

for i, vector in enumerate(X_scaled):
    annoy_index.add_item(i, vector)

# Membangun 10 trees
annoy_index.build(10)

annoy_build_time = time.perf_counter() - start

# Melakukan pencarian
start = time.perf_counter()

annoy_results = []

for q in Q:
    neighbors = annoy_index.get_nns_by_vector(
        q.tolist(),
        k + 1,
        include_distances=True
    )

    annoy_results.append(neighbors[0])

annoy_query_time = (
    time.perf_counter() - start
) / n_query

print("ANNOY berhasil dijalankan!")
print("Build Time:", round(annoy_build_time, 4), "detik")
print("Query Time:", round(annoy_query_time * 1000, 4), "ms/query")

ANNOY berhasil dijalankan!
Build Time: 24.9525 detik
Query Time: 0.1309 ms/query


Langkah 12 – Membuat index FAISS

In [12]:
nlist = 100

# Membuat quantizer
quantizer = faiss.IndexFlatL2(dim)

# Membuat index IVF
faiss_index = faiss.IndexIVFFlat(
    quantizer,
    dim,
    nlist,
    faiss.METRIC_L2
)

# Menentukan jumlah data training
train_size = min(len(X_scaled), max(10000, nlist * 100))

np.random.seed(42)

train_indices = np.random.choice(
    len(X_scaled),
    size=train_size,
    replace=False
)

X_train = X_scaled[train_indices]

# Menghitung waktu pembangunan index
start = time.perf_counter()

faiss_index.train(X_train)
faiss_index.add(X_scaled)

faiss_build_time = time.perf_counter() - start

# Mengatur jumlah cluster yang diperiksa
faiss_index.nprobe = 10

# Melakukan pencarian
start = time.perf_counter()

faiss_distances, faiss_results = faiss_index.search(
    Q,
    k + 1
)

faiss_query_time = (
    time.perf_counter() - start
) / n_query

print("FAISS berhasil dijalankan!")
print("Build Time:", round(faiss_build_time, 4), "detik")
print("Query Time:", round(faiss_query_time * 1000, 4), "ms/query")

FAISS berhasil dijalankan!
Build Time: 0.8719 detik
Query Time: 3.814 ms/query


Langkah 13 – Membuat index HNSW

In [13]:
# Membuat index HNSW
hnsw_index = hnswlib.Index(
    space='l2',
    dim=dim
)

# Menghitung waktu pembangunan index
start = time.perf_counter()

hnsw_index.init_index(
    max_elements=len(X_scaled),
    ef_construction=200,
    M=16
)

hnsw_index.add_items(X_scaled)

hnsw_build_time = time.perf_counter() - start

# Mengatur parameter pencarian
hnsw_index.set_ef(50)

# Melakukan pencarian
start = time.perf_counter()

hnsw_results, hnsw_distances = hnsw_index.knn_query(
    Q,
    k=k + 1
)

hnsw_query_time = (
    time.perf_counter() - start
) / n_query

print("HNSW berhasil dijalankan!")
print("Build Time:", round(hnsw_build_time, 4), "detik")
print("Query Time:", round(hnsw_query_time * 1000, 4), "ms/query")

HNSW berhasil dijalankan!
Build Time: 202.8363 detik
Query Time: 0.1455 ms/query


Langkah 14 – Menghilangkan lagu query dari hasil pencarian

In [14]:
def remove_self(results, query_ids, k=10):

    cleaned = []

    for result, query_id in zip(results, query_ids):

        neighbors = [
            int(idx)
            for idx in result
            if int(idx) >= 0 and int(idx) != int(query_id)
        ]

        cleaned.append(neighbors[:k])

    return cleaned


# Membersihkan hasil Exact NN
exact_clean = remove_self(
    exact_indices,
    query_indices,
    k
)

# Membersihkan hasil ANNOY
annoy_clean = remove_self(
    annoy_results,
    query_indices,
    k
)

# Membersihkan hasil FAISS
faiss_clean = remove_self(
    faiss_results,
    query_indices,
    k
)

# Membersihkan hasil HNSW
hnsw_clean = remove_self(
    hnsw_results,
    query_indices,
    k
)

print("Hasil pencarian berhasil dibersihkan!")

Hasil pencarian berhasil dibersihkan!


Langkah 15 – Menghitung Recall@10

In [15]:
def calculate_recall(exact, approximate, k=10):

    recalls = []

    for exact_result, approx_result in zip(
        exact, approximate
    ):

        exact_set = set(exact_result)
        approx_set = set(approx_result)

        matches = len(exact_set.intersection(approx_set))

        recalls.append(matches / k)

    return np.mean(recalls)


annoy_recall = calculate_recall(
    exact_clean, annoy_clean, k
)

faiss_recall = calculate_recall(
    exact_clean, faiss_clean, k
)

hnsw_recall = calculate_recall(
    exact_clean, hnsw_clean, k
)

print("ANNOY Recall@10:", round(annoy_recall * 100, 2), "%")
print("FAISS Recall@10:", round(faiss_recall * 100, 2), "%")
print("HNSW Recall@10:", round(hnsw_recall * 100, 2), "%")

ANNOY Recall@10: 86.0 %
FAISS Recall@10: 100.0 %
HNSW Recall@10: 100.0 %


Langkah 16 – Membuat tabel perbandingan performa

In [16]:
results = pd.DataFrame({
    "Algoritma": [
        "Exact NN",
        "ANNOY",
        "FAISS",
        "HNSW"
    ],

    "Build Time (detik)": [
        exact_build_time,
        annoy_build_time,
        faiss_build_time,
        hnsw_build_time
    ],

    "Query Time (ms)": [
        exact_query_time * 1000,
        annoy_query_time * 1000,
        faiss_query_time * 1000,
        hnsw_query_time * 1000
    ],

    "Recall@10 (%)": [
        100,
        annoy_recall * 100,
        faiss_recall * 100,
        hnsw_recall * 100
    ]
})

display(
    results.round(4)
)

,Algoritma,Build Time (detik),Query Time (ms),Recall@10 (%)
0,Exact NN,0.0343,57.0846,100.0
1,ANNOY,24.9525,0.1309,86.0
2,FAISS,0.8719,3.8140,100.0
3,HNSW,202.8363,0.1455,100.0


Langkah 17 – Menampilkan 10 lagu terdekat

In [17]:
query_id = int(query_indices[0])

print("LAGU QUERY:")
display(df.iloc[[query_id]])

# Fungsi menampilkan lagu terdekat
def show_tracks(indices, title):

    print("\n", title)

    available_meta = [
        col for col in ['track_name', 'artists', 'track_id']
        if col in df.columns
    ]

    if available_meta:
        display(df.iloc[indices][available_meta])
    else:
        print("Kolom informasi lagu tidak tersedia.")
        print("Indeks lagu:", indices)


# Menampilkan hasil setiap algoritma
show_tracks(exact_clean[0], "EXACT NN")
show_tracks(annoy_clean[0], "ANNOY")
show_tracks(faiss_clean[0], "FAISS")
show_tracks(hnsw_clean[0], "HNSW")

LAGU QUERY:


,artists,danceability,energy,loudness,speechiness,acousticness,instrumentalness,liveness,valence,tempo
174137,Sarah Brightman,0.325,0.291,-11.897,0.0349,0.968,0.109,0.108,0.14,86.843



 EXACT NN


,artists
97012,"Barry Manilow, Sarah Vaughan"
645522,Nobuo Uematsu
97011,"Barry Manilow, Sarah Vaughan"
869738,Alfie Boe
851758,Mario Lanza
413158,Marika Hackman
246126,Libera
417023,Sharon Cuneta
807751,"Tom Waits, Crystal Gayle"
483137,Minako Yoshida



 ANNOY


,artists
97012,"Barry Manilow, Sarah Vaughan"
645522,Nobuo Uematsu
97011,"Barry Manilow, Sarah Vaughan"
869738,Alfie Boe
413158,Marika Hackman
246126,Libera
417023,Sharon Cuneta
483137,Minako Yoshida
655241,Joni Mitchell
822181,Maria Callas



 FAISS


,artists
97012,"Barry Manilow, Sarah Vaughan"
645522,Nobuo Uematsu
97011,"Barry Manilow, Sarah Vaughan"
869738,Alfie Boe
851758,Mario Lanza
413158,Marika Hackman
246126,Libera
417023,Sharon Cuneta
807751,"Tom Waits, Crystal Gayle"
483137,Minako Yoshida



 HNSW


,artists
97012,"Barry Manilow, Sarah Vaughan"
645522,Nobuo Uematsu
97011,"Barry Manilow, Sarah Vaughan"
869738,Alfie Boe
851758,Mario Lanza
413158,Marika Hackman
246126,Libera
417023,Sharon Cuneta
807751,"Tom Waits, Crystal Gayle"
483137,Minako Yoshida


#Kesimpulan

Berdasarkan Hasil Tugas Lab 2 yang sudah dilakukan mengenai Approximate Nearest Neighbors (ANN), telah dilakukan percobaan pencarian lagu terdekat pada dataset Micro Spotify (songs_with_attributes_and_lyrics.csv) berukuran 1,44 GB yang terdiri atas 955.320 baris data bersih (0 data kosong/invalid). Pada praktikum ini digunakan sembilan fitur numerik audio (danceability, energy, loudness, speechiness, acousticness, instrumentalness, liveness, valence, dan tempo) yang dinormalisasi menggunakan StandardScaler agar setiap atribut memiliki skala sebanding dalam perhitungan Euclidean Distance.   

Dari hasil pengujian terhadap 10 sampel query lagu untuk mencari 10 tetangga terdekat ($k=10$), dapat disimpulkan bahwa:

1. Exact NN memberikan akurasi mutlak (Recall@10 = 100,0%) dan waktu inisialisasi tercepat (0,0343 detik), tetapi tidak efisien untuk pencarian real-time berskala besar karena membutuhkan waktu pencarian hingga 57,0846 ms/query.

2. ANNOY (10 trees) menghasilkan waktu pencarian tercepat yaitu 0,1309 ms/query dengan waktu pembangunan indeks 24,9525 detik, namun memiliki tingkat kesesuaian Recall@10 sebesar 86,0%.

3. FAISS (IndexIVFFlat, nlist=100, nprobe=10) memberikan keseimbangan performa yang sangat efisien dengan waktu pembangunan indeks hanya 0,8719 detik, waktu pencarian 3,8140 ms/query, dan akurasi Recall@10 mencapai 100,0%.

4. HNSW (ef_construction=200, M=16, ef=50) memberikan performa pencarian terbaik yang menggabungkan kecepatan sangat tinggi (0,1455 ms/query) dengan akurasi sempurna (Recall@10 = 100,0%), namun membutuhkan waktu pembangunan indeks paling lama (202,8363 detik).

5. Pembersihan indeks hasil pencarian dengan remove_self memastikan evaluasi Recall@10 berjalan objektif karena lagu query itu sendiri (yang memiliki jarak 0) telah dieliminasi dari daftar 10 rekomendasi teratas.

6. Secara keseluruhan, algoritma ANN terbukti mampu mempercepat waktu pencarian ratusan kali lipat dibandingkan Exact NN pada dataset berukuran hampir satu juta baris; HNSW sangat ideal untuk sistem produksi yang mengutamakan kecepatan query dan akurasi tinggi, sementara FAISS sangat cocok apabila indeks sering diperbarui karena waktu build-nya yang jauh lebih singkat.   